In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from scipy import stats

class SpamDetection:
    def __init__(self,algorithms,metrics,data_path,colnames_path):
        self.algorithms=algorithms
        self.metrics=metrics
        self.rank_matrix={}
        self.results={}
        for i in self.metrics:
            self.rank_matrix[i]=[]
            self.results[i]=[]
        with open(colnames_path, 'r') as f:
            col_names=[line.split(':')[0] for line in f.readlines() if ':' in line]
        col_names.append('result')
        data=pd.read_csv(data_path,header=None,names=col_names[1:])
        self.X=data.iloc[:,:-1]
        self.y=data.iloc[:,-1]
        scaler=StandardScaler()
        self.X=scaler.fit_transform(self.X)

    def stratifiedKFold_evaluation(self):
        cv=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)
        for train_index,test_index in cv.split(self.X,self.y):
            for metric in self.metrics:
                resInstance=[]
                for name,model in self.algorithms.items():
                    X_train,X_test=self.X[train_index],self.X[test_index]
                    y_train,y_test=self.y[train_index],self.y[test_index]
                    start_time=time.time()
                    model.fit(X_train,y_train)
                    y_pred=model.predict(X_test)
                    end_time=time.time()
                    if metric==self.metrics[0]:
                        resInstance.append(end_time-start_time)
                    if metric==self.metrics[1]:
                        resInstance.append(accuracy_score(y_test,y_pred))
                    if metric==self.metrics[2]:
                        resInstance.append(f1_score(y_test,y_pred))
                fold_ranks=len(self.algorithms)-np.argsort(np.argsort(resInstance))
                self.rank_matrix[metric].append(fold_ranks)
                self.results[metric].append(resInstance)

    def display_results(self):
        count = 0
        for name, model in self.algorithms.items():
            data = []
            print(f"\ncross validation results: {name}")
            for i in range(len(self.results[self.metrics[0]])):
                foldData = []
                foldData.append(i + 1)
                for metric in self.metrics:
                    foldData.append(self.results[metric][i][count])
                data.append(foldData)
            dataCols = ['fold']
            dataCols.extend(self.metrics)
            data = pd.DataFrame(data, columns=dataCols)
            print(data, end='\n\n')
            count += 1

    def friedman_test(self):
        significance_matrix={}
        for metric in self.metrics:
            rank_mat=np.array(self.rank_matrix[metric])
            k=rank_mat.shape[1]
            N=rank_mat.shape[0]
            rank_sums=np.sum(rank_mat,axis=0)
            squared_rank_sum=np.sum(rank_sums**2)
            friedman_statistic=((12*squared_rank_sum)/(N*k*(k+1)))-(3*N*(k+1))
            df=k-1
            p_value=stats.chi2.cdf(friedman_statistic,df)
            significant=p_value<0.05
            significance_matrix[metric]=significant
            print(f"\nFriedman Test: {metric}")
            print(f"Friedman Statistic:{friedman_statistic}")
            print(f"P-value:{p_value}")
            print(f"Significant:{significant}",end="\n")
            print("-------*-------*-------")
        return significance_matrix

    def nemenyi_test(self,sm):
        for metric,significant in sm.items():
            if (significant==False):
                rank_mat=np.array(self.rank_matrix[metric])
                k=rank_mat.shape[1]
                N=rank_mat.shape[0]
                critical_difference=np.sqrt((k*(k+1))/(6*N))*stats.studentized_range.ppf(0.95,k,float('inf'))
                avg_ranks=np.mean(rank_mat,axis=0)
                pairwise_comparisons={}
                for i in range(k):
                    for j in range(i+1,k):
                        rank_diff=abs(avg_ranks[i]-avg_ranks[j])
                        significantly_different=rank_diff>critical_difference
                        pairwise_comparisons[(i, j)] = {
                            'rank_difference': rank_diff,
                            'critical_difference': critical_difference,
                            'significant': significantly_different
                        }
                print(f"\nNemenyi Test Results: {metric}")
                print("Average Ranks:",avg_ranks)
                print("Critical Difference:",critical_difference)
                print("\nPairwise Comparisons:")
                for (i, j),comparison in pairwise_comparisons.items():
                    print(f"Algorithms {i} vs {j}:")
                    print(f"  Rank Difference:{comparison['rank_difference']}")
                    print(f"  Significant Difference:{comparison['significant']}")
                print("-------*-------*-------")

if __name__=="__main__":
    algorithms={'Logistic Regression':LogisticRegression(),'Support Vector Machine':svm.SVC(random_state=42),'K nearest neighbours':KNeighborsClassifier()}
    metrics=['Training time','accuracy','f1 score']
    data_path=r'spambase.data'
    colnames_path=r'spambase.names'
    SD=SpamDetection(algorithms,metrics,data_path,colnames_path)
    SD.stratifiedKFold_evaluation()
    SD.display_results()
    sm=SD.friedman_test()
    SD.nemenyi_test(sm)